# 05g — Model G Training: Temporal Sequence Models

Treats each driver's race as a **time series**. A sliding window of
the last W laps' features predicts the current lap's `lap_time_ratio`.

10 GPU candidates: GRU/LSTM/TCN/Transformer/CNN variants.
All models Optuna-tunable (no skip).

CV: ExpandingWindowSplit (2019→2020, ..., 2019-23→2024 test).

## 0. Setup

In [ ]:
import os
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    # We're likely in notebooks/ — go up to repo root
    for p in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "pyproject.toml").exists():
            os.chdir(p)
            break

print(f"Working directory: {Path.cwd()}")

In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

from f1_predictor.features.splits import ExpandingWindowSplit, LeaveOneSeasonOut
from f1_predictor.data.storage import (
    load_from_gcs_or_local,
    load_training_parquet,
    save_training_parquet,
    save_model_pickle as gcs_save_model_pickle,
    save_notebook,
    sync_training_from_gcs,
)
from f1_predictor.models.gpu import (
    detect_gpu_backend, get_lightgbm_device, get_torch_device, get_xgboost_device,
)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

TRAINING_DIR = Path("data/training")
MODEL_DIR = Path("data/raw/model")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# GPU detection (supports NVIDIA CUDA and AMD ROCm)
GPU_BACKEND, GPU_NAME = detect_gpu_backend()
TORCH_DEVICE = get_torch_device()
print(f"GPU backend: {GPU_BACKEND} ({GPU_NAME})")
print(f"PyTorch device: {TORCH_DEVICE}")

# Deep learning models (PyTorch — works on both CUDA and ROCm via HIP)
DL_AVAILABLE = False
try:
    from f1_predictor.models.architectures import GRU2Layer, FTTransformerWrapper, MLP3Layer
    DL_AVAILABLE = TORCH_DEVICE != "cpu"
    print(f"DL models available: {DL_AVAILABLE}")
except (ImportError, NameError):
    print("DL models not available (torch/rtdl not installed)")

In [ ]:
def save_predictions(model, X, y, id_df, model_type, model_name, split_name):
    """Save prediction parquet locally and to GCS."""
    preds = model.predict(X)
    out = id_df.copy()
    out["y_true"] = y.values
    out["y_pred"] = preds
    fname = f"model_{model_type}_{model_name}_{split_name}.parquet"
    uri = save_training_parquet(out, fname, TRAINING_DIR)
    print(f"  Saved {fname} -> {uri}")
    return preds


def save_model_pkl(model, model_type, model_name):
    """Save model pickle locally and to GCS."""
    fname = f"Model_{model_type}_{model_name}.pkl"
    uri = gcs_save_model_pickle(model, fname, MODEL_DIR)
    print(f"  Saved {fname} -> {uri}")

## 1. Build Sequence Training Data

In [ ]:
from f1_predictor.features.simulation_features import (
    build_simulation_training_data,
    SIMULATION_FEATURE_COLS,
)
from f1_predictor.features.sequence_features import (
    build_sequence_training_data,
    slice_window,
)

laps = load_from_gcs_or_local(
    "data/raw/laps/all_laps.parquet",
    Path("data/raw/laps/all_laps.parquet"),
)
races = load_from_gcs_or_local(
    "data/raw/race/all_races.parquet",
    Path("data/raw/race/all_races.parquet"),
)

# Build tabular simulation data first
df_sim = build_simulation_training_data(laps, races)
print(f"Tabular shape: {df_sim.shape}")

# Reshape into windowed sequences (max_window=10, sliced per candidate)
MAX_WINDOW = 10
X_seq, y_seq, id_df = build_sequence_training_data(df_sim, max_window=MAX_WINDOW)
print(f"X_seq shape: {X_seq.shape}  (samples, window, features+mask)")
print(f"y shape: {y_seq.shape}")
print(f"Target stats:\n{pd.Series(y_seq).describe()}")

In [ ]:
# seasons for CV split — use id_df
groups = id_df["season"].values
n_features_seq = X_seq.shape[2]  # includes mask channel
print(f"Sequence features (incl mask): {n_features_seq}")
print(f"Seasons: {sorted(set(groups))}")

## 2. CV Splitter

In [ ]:
splitter = ExpandingWindowSplit(
    fold_definitions=[
        ([2019], 2020),
        ([2019, 2020], 2021),
        ([2019, 2020, 2021], 2022),
        ([2019, 2020, 2021, 2022], 2023),
    ],
    test_season=2024,
)
print(f"CV folds: {splitter.get_n_splits()}")
for i, (tr, va) in enumerate(splitter.split(groups)):
    tr_seasons = sorted(set(groups[tr]))
    va_seasons = sorted(set(groups[va]))
    print(f"  Fold {i}: train seasons={tr_seasons}, val seasons={va_seasons}, "
          f"train={len(tr):,}, val={len(va):,}")

## 3. Model Candidates and Helpers

10 sequence model candidates, all Optuna-tunable.

In [ ]:
from f1_predictor.models.sequence_architectures import (
    SeqGRU_Shallow, SeqGRU_Deep, SeqGRU_Bidir,
    SeqLSTM_Shallow, SeqLSTM_Deep, SeqLSTM_Bidir,
    SeqTCN, SeqTransformer, SeqGRU_Attn, SeqCNN1D,
)

DL_SKIP_OPTUNA = set()  # all models tunable
NAN_TOLERANT = set()  # no tree-based models

MODEL_CLASSES_G = {
    "SeqGRU_Shallow": SeqGRU_Shallow,
    "SeqGRU_Deep": SeqGRU_Deep,
    "SeqGRU_Bidir": SeqGRU_Bidir,
    "SeqLSTM_Shallow": SeqLSTM_Shallow,
    "SeqLSTM_Deep": SeqLSTM_Deep,
    "SeqLSTM_Bidir": SeqLSTM_Bidir,
    "SeqTCN": SeqTCN,
    "SeqTransformer": SeqTransformer,
    "SeqGRU_Attn": SeqGRU_Attn,
    "SeqCNN1D": SeqCNN1D,
}


def get_candidates_g():
    candidates = {}
    if not DL_AVAILABLE:
        print("WARNING: No GPU — sequence models require PyTorch with CUDA/ROCm")
        return candidates
    for name, cls in MODEL_CLASSES_G.items():
        candidates[name] = cls(n_features=n_features_seq)
    print(f"Candidates ({len(candidates)}): {list(candidates.keys())}")
    return candidates


def _seq_optuna_space(trial, model_name):
    params = {
        "n_features": n_features_seq,
        "window_size": trial.suggest_int("window_size", 3, MAX_WINDOW),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128, 256]),
        "dropout": trial.suggest_float("dropout", 0.05, 0.5),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [512, 1024, 2048]),
    }
    if model_name in ("SeqGRU_Shallow", "SeqGRU_Deep", "SeqGRU_Bidir",
                       "SeqLSTM_Shallow", "SeqLSTM_Deep", "SeqLSTM_Bidir",
                       "SeqGRU_Attn"):
        params["num_layers"] = trial.suggest_int("num_layers", 1, 4)
    if model_name in ("SeqTCN", "SeqCNN1D"):
        params["kernel_size"] = trial.suggest_categorical("kernel_size", [3, 5, 7])
    if model_name == "SeqTCN":
        params["num_layers"] = trial.suggest_int("num_layers", 2, 6)
    if model_name == "SeqTransformer":
        params["num_layers"] = trial.suggest_int("num_layers", 1, 4)
        params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4, 8])
        # d_model must be divisible by n_heads
        hd = params["hidden_dim"]
        nh = params["n_heads"]
        if hd % nh != 0:
            params["hidden_dim"] = max(nh, (hd // nh) * nh)
    return params


def get_optuna_param_space_g(name, trial):
    return _seq_optuna_space(trial, name)


def reconstruct_params_g(name, best_params):
    params = dict(best_params)
    params["n_features"] = n_features_seq
    return params

In [ ]:
def cv_evaluate_g(model, X_seq_full, y_full, splitter, groups):
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    rmses, maes = [], []
    for tr_idx, va_idx in splitter.split(groups):
        import sklearn.base
        m = sklearn.base.clone(model)
        X_tr, y_tr = X_seq_full[tr_idx], y_full[tr_idx]
        X_va, y_va = X_seq_full[va_idx], y_full[va_idx]
        m.fit(X_tr, y_tr)
        preds = m.predict(X_va)
        rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        maes.append(mean_absolute_error(y_va, preds))
    return {"mean_rmse": np.mean(rmses), "std_rmse": np.std(rmses), "mean_mae": np.mean(maes)}


def screen_models_g(candidates, X_seq_full, y_full, splitter, groups):
    rows = []
    for name, model in candidates.items():
        try:
            result = cv_evaluate_g(model, X_seq_full, y_full, splitter, groups)
            rows.append({"model": name, **result})
            print(f"  {name}: RMSE={result['mean_rmse']:.4f}")
        except Exception as e:
            print(f"  {name}: FAILED — {e}")
    return pd.DataFrame(rows).sort_values("mean_rmse").reset_index(drop=True)


def run_optuna_round_g(name, X_seq_full, y_full, splitter, groups, n_trials):
    def objective(trial):
        params = get_optuna_param_space_g(name, trial)
        ws = params.pop("window_size", MAX_WINDOW)
        model_cls = MODEL_CLASSES_G[name]
        model = model_cls(**params, window_size=ws)
        # Slice sequences to candidate's window size
        X_sliced = slice_window(X_seq_full, ws)
        result = cv_evaluate_g(model, X_sliced, y_full, splitter, groups)
        return result["mean_rmse"]
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(
        objective, n_trials=n_trials,
        catch=(Exception,), show_progress_bar=False,
    )
    return study.best_params, study.best_value

## 4. Round 1 — Screen Models

In [ ]:
candidates = get_candidates_g()
r1_results = screen_models_g(candidates, X_seq, y_seq, splitter, groups)
r1_results[["model", "mean_rmse", "std_rmse", "mean_mae"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(r1_results["model"], r1_results["mean_rmse"], xerr=r1_results["std_rmse"])
ax.set_xlabel("Mean RMSE (CV)")
ax.set_title("Round 1: Model Screening — Model G (Temporal Sequences)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
top7_names = r1_results["model"].head(7).tolist()
print(f"Advancing to Round 2: {top7_names}")
eliminated = r1_results["model"].iloc[7:].tolist()
print(f"Eliminated: {eliminated}")

## 5. Round 2 — Optuna (top 7, 10 trials each)

In [ ]:
r2_results = []
for name in top7_names:
    print(f"Tuning {name}...")
    best_params, best_rmse = run_optuna_round_g(
        name, X_seq, y_seq, splitter, groups, n_trials=10)
    r2_results.append({"model": name, "best_rmse": best_rmse, "best_params": best_params})
    print(f"  Best RMSE: {best_rmse:.4f}")

r2_df = pd.DataFrame(r2_results).sort_values("best_rmse").reset_index(drop=True)
r2_df[["model", "best_rmse"]]

In [ ]:
top5_names = r2_df["model"].head(5).tolist()
r2_best_params = {row["model"]: row["best_params"] for _, row in r2_df.iterrows()}
print(f"Advancing to Round 3: {top5_names}")

## 6. Round 3 — Final Tuning (top 5, 15 trials each)

In [ ]:
r3_results = []
for name in top5_names:
    print(f"Fine-tuning {name}...")
    best_params, best_rmse = run_optuna_round_g(
        name, X_seq, y_seq, splitter, groups, n_trials=15)
    r3_results.append({"model": name, "best_rmse": best_rmse, "best_params": best_params})
    print(f"  Best RMSE: {best_rmse:.4f}")

r3_df = pd.DataFrame(r3_results).sort_values("best_rmse").reset_index(drop=True)
r3_best_params = {row["model"]: row["best_params"] for _, row in r3_df.iterrows()}
r3_df[["model", "best_rmse"]]

## 7. Test Set Evaluation (Per-Lap)

In [ ]:
train_idx, test_idx = splitter.get_test_split(groups)
X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]
y_train_full, y_test = y_seq[train_idx], y_seq[test_idx]
id_train, id_test = id_df.iloc[train_idx], id_df.iloc[test_idx]

print(f"Train: {X_seq_train.shape}, Test: {X_seq_test.shape}")
print(f"Test season(s): {sorted(set(groups[test_idx]))}")

In [ ]:
final_results = []
for name in top5_names:
    params = reconstruct_params_g(name, r3_best_params[name])
    ws = params.pop("window_size", MAX_WINDOW)
    model_cls = MODEL_CLASSES_G[name]
    model = model_cls(**params, window_size=ws)

    X_tr_sliced = slice_window(X_seq_train, ws)
    X_te_sliced = slice_window(X_seq_test, ws)

    model.fit(X_tr_sliced, y_train_full)

    train_preds = model.predict(X_tr_sliced)
    train_rmse = np.sqrt(mean_squared_error(y_train_full, train_preds))

    test_preds = model.predict(X_te_sliced)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

    val_rmse = r3_df.loc[r3_df["model"] == name, "best_rmse"].values[0]

    final_results.append({
        "model": name,
        "train_rmse": train_rmse, "val_rmse": val_rmse,
        "test_rmse": test_rmse, "overfit_gap": test_rmse - val_rmse,
        "window_size": ws,
    })
    print(f"{name} (ws={ws}): train_rmse={train_rmse:.4f}, "
          f"val_rmse={val_rmse:.4f}, test_rmse={test_rmse:.4f}")

final_df = pd.DataFrame(final_results).sort_values("test_rmse").reset_index(drop=True)
final_df

## 8. Sequence Simulation (2024 Test Races)

In [ ]:
from f1_predictor.simulation.sequence_simulator import SequenceRaceSimulator
from f1_predictor.simulation.defaults import build_circuit_defaults
from f1_predictor.simulation.evaluation import evaluate_simulation
from f1_predictor.features.race_features import LOCATION_ALIASES

circuit_defaults = build_circuit_defaults(laps)

# Retrain best model on full training set
best_row = final_df.iloc[0]
best_model_name = best_row["model"]
best_ws = int(best_row["window_size"])
best_params = reconstruct_params_g(best_model_name, r3_best_params[best_model_name])
best_params.pop("window_size", None)
best_model = MODEL_CLASSES_G[best_model_name](**best_params, window_size=best_ws)
X_tr_sliced = slice_window(X_seq_train, best_ws)
best_model.fit(X_tr_sliced, y_train_full)

seq_sim = SequenceRaceSimulator(best_model, circuit_defaults, window_size=best_ws)
print(f"Sequence simulator: {best_model_name} (window={best_ws})")

In [ ]:
test_races = races[races["season"] == 2024].copy()
test_race_list = test_races.groupby(
    ["season", "round", "event_name"]
).first().reset_index()

sim_results = []
for _, race_row in test_race_list.iterrows():
    event = race_row["event_name"]
    event_norm = LOCATION_ALIASES.get(event, event)

    if event_norm not in circuit_defaults:
        print(f"  Skipping {event} (no circuit data)")
        continue

    race_drivers = test_races[
        (test_races["season"] == race_row["season"])
        & (test_races["round"] == race_row["round"])
    ].copy()

    drivers_input = []
    actual_positions = {}
    for _, drv in race_drivers.iterrows():
        q1 = drv.get("q1_time_sec")
        q2 = drv.get("q2_time_sec")
        q3 = drv.get("q3_time_sec")
        q_times = [t for t in [q1, q2, q3] if pd.notna(t)]
        if not q_times or pd.isna(drv.get("grid_position")):
            continue
        drivers_input.append({
            "driver": drv["driver_abbrev"],
            "grid_position": int(drv["grid_position"]),
            "q1": q1 if pd.notna(q1) else None,
            "q2": q2 if pd.notna(q2) else None,
            "q3": q3 if pd.notna(q3) else None,
            "initial_tyre": "MEDIUM",
        })
        if pd.notna(drv.get("finish_position")):
            actual_positions[drv["driver_abbrev"]] = int(drv["finish_position"])

    if len(drivers_input) < 10:
        continue

    try:
        result = seq_sim.simulate(event_norm, drivers_input)
        for fr in result.final_results:
            if fr["driver"] in actual_positions:
                sim_results.append({
                    "event": event, "driver": fr["driver"],
                    "predicted_pos": fr["position"],
                    "actual_pos": actual_positions[fr["driver"]],
                })
        print(f"  {event}: simulated {len(drivers_input)} drivers")
    except Exception as e:
        print(f"  {event}: failed — {e}")

sim_df = pd.DataFrame(sim_results)
print(f"\nSimulation results: {len(sim_df)} driver-race predictions")

In [ ]:
if len(sim_df) > 0:
    sim_metrics = evaluate_simulation(sim_df)
    print("=" * 60)
    print("MODEL G — SEQUENCE SIMULATION (2024)")
    print("=" * 60)
    for k, v in sim_metrics.items():
        print(f"  {k:20s}: {v}")

## 9. Save Artifacts

In [ ]:
ID_COLS_SEQ = ["season", "round", "driver_abbrev", "lap_number"]

for name in top5_names:
    params = reconstruct_params_g(name, r3_best_params[name])
    ws = params.pop("window_size", MAX_WINDOW)
    model_cls = MODEL_CLASSES_G[name]
    model = model_cls(**params, window_size=ws)

    X_tr_s = slice_window(X_seq_train, ws)
    X_te_s = slice_window(X_seq_test, ws)
    model.fit(X_tr_s, y_train_full)

    # Training predictions
    train_preds = model.predict(X_tr_s)
    out = id_train.copy()
    out["y_true"] = y_train_full
    out["y_pred"] = train_preds
    fname = f"model_G_{name}_Training.parquet"
    uri = save_training_parquet(out, fname, TRAINING_DIR)
    print(f"  Saved {fname} -> {uri}")

    # Test predictions
    test_preds = model.predict(X_te_s)
    out = id_test.copy()
    out["y_true"] = y_test
    out["y_pred"] = test_preds
    fname = f"model_G_{name}_Test.parquet"
    uri = save_training_parquet(out, fname, TRAINING_DIR)
    print(f"  Saved {fname} -> {uri}")

    # OOF validation predictions
    oof_preds = np.full(len(y_seq), np.nan)
    for tr_idx, va_idx in splitter.split(groups):
        import sklearn.base
        fold_model = sklearn.base.clone(model)
        X_tr_fold = slice_window(X_seq[tr_idx], ws)
        fold_model.fit(X_tr_fold, y_seq[tr_idx])
        X_va_fold = slice_window(X_seq[va_idx], ws)
        oof_preds[va_idx] = fold_model.predict(X_va_fold)

    val_mask = ~np.isnan(oof_preds)
    val_out = id_df.loc[val_mask].copy()
    val_out["y_true"] = y_seq[val_mask]
    val_out["y_pred"] = oof_preds[val_mask]
    fname = f"model_G_{name}_Validation.parquet"
    uri = save_training_parquet(val_out, fname, TRAINING_DIR)
    print(f"  Saved {fname} -> {uri}")

    save_model_pkl(model, "G", name)

print("\nDone! All Model G artifacts saved.")

## Summary

In [ ]:
print("=" * 60)
print("MODEL G (TEMPORAL SEQUENCE) TRAINING COMPLETE")
print("=" * 60)
print(f"\nPer-lap evaluation (top 5, sorted by test RMSE):")
for _, row in final_df.iterrows():
    print(f"  {row['model']:25s} ws={int(row['window_size'])} "
          f"test_rmse={row['test_rmse']:.6f} gap={row['overfit_gap']:.6f}")

if len(sim_df) > 0:
    print(f"\nSequence simulation (2024):")
    print(f"  Position RMSE: {sim_metrics['position_rmse']:.4f}")
    print(f"  Spearman:      {sim_metrics['spearman_mean']:.4f}")
    print(f"  Within-3:      {sim_metrics['within_3']:.1f}%")
print(f"\nArtifacts saved to:")
print(f"  Predictions: {TRAINING_DIR}")
print(f"  Models: {MODEL_DIR}")